<a href="https://colab.research.google.com/github/nmwaura4/Projects/blob/main/Biological_supply_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import re
import string
# Natural Language Toolkit (NLTK)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression


# TextBlob for Sentiment Analysis
from textblob import TextBlob

In [ ]:
df=pd.read_excel('/content/Supply_Data.xlsx')
df.head()

In [ ]:
df.shape

In [ ]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text=re.sub(r'\s+', ' ', text)
    text=re.sub(r'\d+', '', text)
    text=re.sub(r'http\S+', '', text)
    text=re.sub(r'@\w+', '', text)
    text=re.sub(r'#\w+', '', text)
    text=re.sub(r'\s+', ' ', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

In [ ]:
stop_words = set(stopwords.words('english'))

stemmer = PorterStemmer()

lemmatizer = WordNetLemmatizer()

In [ ]:


text = "Biological pesticide worked perfectly 😊🌱💯"

clean_text = re.sub(r'[^\x00-\x7F]+', '', text)

print(clean_text)

In [ ]:
df['clean'] = df[df.columns[0]].apply(lambda x: clean_text(x))

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
df['clean']=df['clean'].apply(lambda x: word_tokenize(x))

In [ ]:
df['clean']=df['clean'].apply(lambda x: [word for word in x if word not in stopwords.words('english')])

In [ ]:
df['polarity']=df['clean'].apply(lambda x: TextBlob(' '.join(x)).sentiment.polarity)

In [ ]:
df['sentment']=df['polarity'].apply(lambda x: 'positive' if x>0 else ('negative' if x<0 else 'neutral'))

In [ ]:
df['sentment'].value_counts().plot(kind='bar')
plt.title('sentiment distribution')
plt.xlabel('sentiment')
plt.ylabel('counts')
plt.show()

In [ ]:
negative_keywords = [
    "destroying",
    "strange",
    "leaf burns",
    "expensive",
    "leaked",
    "didn’t see any improvement",
    "confusing",
    "broken",
    "complaint",
    "returned",
    "terrible",
    "wrong product",
    "taking too long",
    "no effect",
    "missing",
    "multiplied",
    "stopped working",
    "disappointed",
    "waited",
    "damaged",
    "resistant",
    "ignored",
    "diluted",
    "ineffective",
    "strong smell",
    "regret",
    "rude",
    "still have aphids",
    "broke immediately",
    "short shelf life",
    "never updated",
    "delayed",
    "yellow spots",
    "half empty",
    "weak effectiveness",
    "didn’t work",
    "expired",
    "too quickly",
    "no visible reduction",
    "delay",
    "problem"
]

In [ ]:
positive_keywords = [
    "perfectly",
    "on time",
    "fewer",
    "effective",
    "properly",
    "secure",
    "professional",
    "healthier",
    "reduced",
    "very effective",
    "eco-friendly",
    "significantly",
    "improved",
    "simple",
    "fast",
    "impressive",
    "mixed well",
    "excellent",
    "recommend",
    "organic",
    "good condition",
    "better",
    "answered",
    "affordable",
    "easy to use",
    "quality",
    "recovered",
    "good experience",
    "advantage",
    "exactly as described",
    "safe",
    "reliable",
    "great",
    "worked very well",
    "helped control",
    "suitable",
    "excellent pest reduction",
    "neat",
    "disappeared",
    "exceeded expectations",
    "look healthier",
    "reduced chemical usage",
    "easy to store",
    "resolved",
    "controlled",
    "showed improvement",
    "feel safer",
    "healthy",
    "success",
    "quickly",
    "efficient",
    "strong results",
    "high quality",
    "trusted",
    "effective control",
    "safe around pets",
    "premium",
    "best",
    "fantastic",
    "awesome",
    "smooth",
    "clean",
    "stable",
    "durable",
    "fresh",
    "positive",
    "satisfied",
    "happy",
    "love",
    "amazing",
    "wonderful",
    "pleasant",
    "useful",
    "valuable",
    "supportive",
    "responsive",
    "powerful",
    "productive"
]

In [ ]:
positive_sentiment=df[df['sentment']=='positive']
negative_sentiment=df[df['sentment']=='negative']
neutral_sentiment=df[df['sentment']=='neutral']
positive_sentiment.head(10)
negative_sentiment.head(10)
neutral_sentiment.head()

In [ ]:
positive_sentiment.head(10)

In [ ]:
def chek_negative_keywords(text):
  for word in negative_keywords:
    if word in text:
      return True
  return False

In [ ]:
df['has_negative_keywords']=df['clean'].apply(lambda x: chek_negative_keywords(' '.join(x)))

In [ ]:
df['has_positive_keywords']=df['clean'].apply(lambda x: chek_negative_keywords(' '.join(x)))
df['has_neutral_keywords']=df['clean'].apply(lambda x: chek_negative_keywords(' '.join(x)))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(x=df['sentment'].value_counts().index,y=df['sentment'].value_counts().values)
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()
#

In [ ]:
from collections import Counter

# Flatten the list of cleaned words for negative sentiment and count keywords
all_negative_words = [word for sublist in negative_sentiment['clean'] for word in sublist]
negative_keyword_counts = Counter()
for word in all_negative_words:
    for keyword in negative_keywords:
        if keyword in word: # Check if the keyword is a substring of the word
            negative_keyword_counts[keyword] += 1

# Create a DataFrame from the counts
keyword_df = pd.DataFrame(negative_keyword_counts.most_common(), columns=['keyword', 'count'])

plt.figure(figsize=(10,6))
sns.barplot(x='count',y='keyword',data=keyword_df.head(20),palette='viridis')
plt.title('Top 20 Negative Keywords')
plt.xlabel('Count')
plt.ylabel('Keyword')
plt.show()

In [ ]:
def check_keywords(text, keyword_list):
    for keyword in keyword_list:
        if keyword in text:
            return True
    return False

df['keyword_sentiment'] = 'neutral'
df.loc[df['clean'].apply(lambda x: check_keywords(' '.join(x), positive_keywords)), 'keyword_sentiment'] = 'positive'
df.loc[df['clean'].apply(lambda x: check_keywords(' '.join(x), negative_keywords)), 'keyword_sentiment'] = 'negative'

# Prioritize negative if both positive and negative keywords are present
df.loc[df['clean'].apply(lambda x: check_keywords(' '.join(x), positive_keywords) and check_keywords(' '.join(x), negative_keywords)), 'keyword_sentiment'] = 'negative'

# Filter out neutral sentiment for confusion matrix if you only want to compare positive/negative
# Or, keep all three for a 3x3 matrix

# Let's compare TextBlob sentiment ('sentment') with our new keyword-based sentiment ('keyword_sentiment')
# Ensure both columns are of string type for consistent comparison
df['sentment'] = df['sentment'].astype(str)
df['keyword_sentiment'] = df['keyword_sentiment'].astype(str)

# Get unique classes to ensure the confusion matrix has consistent ordering
labels = sorted(df['sentment'].unique().tolist())

cm = confusion_matrix(df['sentment'], df['keyword_sentiment'], labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Keyword-based Sentiment')
plt.ylabel('TextBlob Sentiment')
plt.title('Confusion Matrix: TextBlob vs. Keyword-based Sentiment')
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report
report = classification_report(df['sentment'], df['keyword_sentiment'], labels=labels)

print("Classification Report:\n", report)

In [ ]:
X=df['clean']
y=df['sentment']

In [ ]:
# Join the list of words back into strings for vectorization
X_joined = X.apply(lambda x: ' '.join(x))

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the text data
X_vectorized = vectorizer.fit_transform(X_joined)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

# Initialize and train the RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

In [ ]:
#from sklearn.metrics import classification_report

# Generate the classification report
report = classification_report(df['sentment'], df['keyword_sentiment'], labels=labels)

print("Classification Report:\n", report)